In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import albumentations as A
from collections import deque
import datetime

# Constants
SEQUENCE_LENGTH = 20
IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 100

class VideoDataset:
    def __init__(self, data_path, sequence_length=20, image_size=(128, 128)):
        self.data_path = data_path
        self.sequence_length = sequence_length
        self.image_size = image_size
        self.augmenter = self.get_augmenter()
        
    def get_augmenter(self):
        return A.Compose([
            A.RandomBrightnessContrast(p=0.3),
            A.RandomGamma(p=0.2),
            A.GaussNoise(p=0.2),
            A.HorizontalFlip(p=0.3),
            A.OneOf([
                A.MotionBlur(p=0.2),
                A.MedianBlur(blur_limit=3, p=0.2),
                A.GaussianBlur(blur_limit=3, p=0.2),
            ], p=0.2),
            A.OneOf([
                A.RandomRotate90(p=0.2),
                A.RandomResizedCrop(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, p=0.2),
            ], p=0.2),
        ])
    
    def load_video_frames(self, video_path, augment=True):
        frames = []
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = np.linspace(0, total_frames-1, self.sequence_length, dtype=int)
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, self.image_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if augment:
                    frame = self.augmenter(image=frame)['image']
                frames.append(frame)
            else:
                frames.append(np.zeros((self.image_size[0], self.image_size[1], 3)))
        
        cap.release()
        return np.array(frames) / 255.0
    
    def create_dataset(self, augment=True):
        features = []
        labels = []
        
        for activity_type in os.listdir(self.data_path):
            activity_path = os.path.join(self.data_path, activity_type)
            
            if activity_type == 'Normal':
                for video_file in os.listdir(activity_path):
                    if video_file.endswith(('.mp4', '.avi')):
                        video_path = os.path.join(activity_path, video_file)
                        frames = self.load_video_frames(video_path, augment)
                        features.append(frames)
                        labels.append('Normal')
                        
                        # Add augmented version for normal videos
                        if augment:
                            frames_aug = self.load_video_frames(video_path, augment)
                            features.append(frames_aug)
                            labels.append('Normal')
            
            elif activity_type == 'Abnormal':
                for category in os.listdir(activity_path):
                    category_path = os.path.join(activity_path, category)
                    if os.path.isdir(category_path):
                        for video_file in os.listdir(category_path):
                            if video_file.endswith(('.mp4', '.avi')):
                                video_path = os.path.join(category_path, video_file)
                                frames = self.load_video_frames(video_path, augment)
                                features.append(frames)
                                labels.append(f'Abnormal_{category}')
        
        return np.array(features), np.array(labels)

class ModelTrainer:
    def __init__(self, model_type, num_classes):
        self.model_type = model_type
        self.num_classes = num_classes
        self.model = self.build_model()
        
    def build_model(self):
        if self.model_type == 'convlstm':
            return create_enhanced_convlstm_model(
                sequence_length=SEQUENCE_LENGTH,
                image_height=IMAGE_HEIGHT,
                image_width=IMAGE_WIDTH,
                channels=CHANNELS,
                num_classes=self.num_classes
            )
        else:
            return create_enhanced_lrcn_model(
                sequence_length=SEQUENCE_LENGTH,
                image_height=IMAGE_HEIGHT,
                image_width=IMAGE_WIDTH,
                channels=CHANNELS,
                num_classes=self.num_classes
            )
    
    def compile_model(self):
        self.model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy', 
                    tf.keras.metrics.Precision(), 
                    tf.keras.metrics.Recall(),
                    tf.keras.metrics.AUC()]
        )
    
    def train_and_validate(self, X_train, y_train, X_val, y_val):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = f'{self.model_type}_model_{timestamp}'
        
        callbacks = [
            ModelCheckpoint(
                f'{model_name}_best.h5',
                monitor='val_loss',
                save_best_only=True,
                mode='min'
            ),
            EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.2,
                patience=5,
                min_lr=1e-6
            )
        ]
        
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks
        )
        
        return history, model_name

class ModelEvaluator:
    @staticmethod
    def plot_training_history(history, model_name):
        metrics = ['loss', 'accuracy', 'precision', 'recall', 'auc']
        plt.figure(figsize=(15, 10))
        
        for i, metric in enumerate(metrics, 1):
            plt.subplot(2, 3, i)
            plt.plot(history.history[metric], label=f'Training {metric}')
            plt.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
            plt.title(f'{model_name} - {metric.capitalize()}')
            plt.xlabel('Epoch')
            plt.ylabel(metric.capitalize())
            plt.legend()
        
        plt.tight_layout()
        plt.savefig(f'{model_name}_training_history.png')
        plt.close()
    
    @staticmethod
    def evaluate_model(model, X_test, y_test, label_encoder, model_name):
        # Get predictions
        y_pred = model.predict(X_test)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true_classes = np.argmax(y_test, axis=1)
        
        # Create classification report
        class_names = label_encoder.classes_
        report = classification_report(y_true_classes, y_pred_classes, 
                                    target_names=class_names, output_dict=True)
        
        # Plot confusion matrix
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_true_classes, y_pred_classes)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names)
        plt.title(f'{model_name} - Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        plt.savefig(f'{model_name}_confusion_matrix.png')
        plt.close()
        
        return report

def main():
    # Initialize dataset
    dataset = VideoDataset('path/to/dataset')
    features, labels = dataset.create_dataset()
    
    # Encode labels
    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(labels)
    one_hot_labels = tf.keras.utils.to_categorical(encoded_labels)
    
    # Split dataset
    X_train, X_temp, y_train, y_temp = train_test_split(
        features, one_hot_labels, test_size=0.3, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )
    
    # Train and evaluate both models
    models = ['convlstm', 'lrcn']
    results = {}
    
    for model_type in models:
        print(f"\nTraining {model_type.upper()} model...")
        trainer = ModelTrainer(model_type, len(label_encoder.classes_))
        trainer.compile_model()
        
        history, model_name = trainer.train_and_validate(
            X_train, y_train, X_val, y_val
        )
        
        # Evaluate model
        evaluator = ModelEvaluator()
        evaluator.plot_training_history(history, model_name)
        
        report = evaluator.evaluate_model(
            trainer.model, X_test, y_test, label_encoder, model_name
        )
        
        results[model_type] = {
            'history': history.history,
            'report': report,
            'model_name': model_name
        }
        
        # Save model and results
        trainer.model.save(f'{model_name}_final.h5')
    
    # Compare models
    print("\nModel Comparison:")
    for model_type in results:
        print(f"\n{model_type.upper()} Results:")
        report = results[model_type]['report']
        print(f"Accuracy: {report['accuracy']:.4f}")
        print(f"Weighted Avg F1-Score: {report['weighted avg']['f1-score']:.4f}")

if __name__ == "__main__":
    main()